# 06 - Statistical Tests & Effect Sizes

**Purpose:**
- Address **Reviewer C3** by computing practical effect sizes (not just p-values) to quantify the magnitude of differences between normal and anomalous providers.
- Numerical features: Mann-Whitney U test + **Rank-Biserial Correlation ($r$)**.
- Categorical features: Chi-Square Test of Independence + **Cramér's V**.
- **Audit Fixes:** 
  - Fix BUG-03: Print and export the exact $U$ and $\chi^2$ statistics to CSV.
  - Fix BUG-04: Use exact, unambiguous group names matching the manuscript ("IF-Only vs. LOF-Only").

**Inputs:**
- `data/raw/healthcare_providers.csv` (to access unscaled numericals and categoricals)
- `data/processed/anomaly_labels.parquet`

**Outputs:**
- `outputs/tables/table3_mannwhitney_effectsize.csv`
- `outputs/tables/table4_chisquare_effectsize.csv`\n

In [ ]:
# Cell 01: Mount Storage & Bootstrap Paths
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Paper1_Revision')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

for folder in ['data/raw', 'data/interim', 'data/processed', 
               'outputs/figures', 'outputs/tables', 'outputs/models', 
               'outputs/notebook_exports']:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
    
print(f"Base directory set to: {BASE_DIR}")\n

In [ ]:
# Cell 02: Imports, Global Seeds & Style
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, chi2_contingency
import warnings

warnings.filterwarnings('ignore')\n

In [ ]:
# Cell 03: Load Raw Data & Labels
raw_data_path = BASE_DIR / 'data' / 'raw' / 'healthcare_providers.csv'
labels_path = BASE_DIR / 'data' / 'processed' / 'anomaly_labels.parquet'

df_raw = pd.read_csv(raw_data_path, low_memory=False)
df_labels = pd.read_parquet(labels_path)

assert len(df_raw) == len(df_labels), "Row mismatch between raw data and labels!"

# Merge labels into raw data for slicing
df = pd.concat([df_raw, df_labels], axis=1)

# Define the 7 numerical features used for Mann-Whitney
num_features = [
    'Number of Services',
    'Number of Medicare Beneficiaries',
    'Number of Distinct Medicare Beneficiary/Per Day Services',
    'Average Medicare Allowed Amount',
    'Average Submitted Charge Amount',
    'Average Medicare Payment Amount',
    'Average Medicare Standardized Amount'
]

# Identify categorical features for Chi-Square (e.g., Provider Type, State)
cat_features = ['Provider Type', 'State Code of the Provider']\n

## 1. Define Mutually Exclusive Comparison Groups
To match Manuscript Table 3 & 4 (Fixing BUG-04).\n

In [ ]:
# Cell 04: Define Groups
# Normal = 1, Anomaly = -1
group_normal = df[(df['IF_Label'] == 1) & (df['LOF_05_Label'] == 1)]
group_if_only = df[(df['IF_Label'] == -1) & (df['LOF_05_Label'] == 1)]
group_lof_only = df[(df['IF_Label'] == 1) & (df['LOF_05_Label'] == -1)]
group_overlap = df[(df['IF_Label'] == -1) & (df['LOF_05_Label'] == -1)]

print(f"Group: Normal (Both)         n = {len(group_normal)}")
print(f"Group: IF-Only Anomalies     n = {len(group_if_only)}")
print(f"Group: LOF-Only Anomalies    n = {len(group_lof_only)}")
print(f"Group: Overlap (Both -1)     n = {len(group_overlap)}")\n

## 2. Mann-Whitney U & Rank-Biserial Correlation (Numerical Features)\n

In [ ]:
# Cell 05: Run Mann-Whitney U Tests
def run_mwu_tests(group_a, group_b, group_name_a, group_name_b):
    results = []
    n1 = len(group_a)
    n2 = len(group_b)
    
    for col in num_features:
        # Drop NaNs for the test
        x = group_a[col].dropna()
        y = group_b[col].dropna()
        
        # Mann-Whitney U test
        u_stat, p_val = mannwhitneyu(x, y, alternative='two-sided')
        
        # Rank-Biserial Correlation: r = 1 - (2U / (n1 * n2))
        r = 1 - (2 * u_stat / (n1 * n2))
        
        results.append({
            'Comparison': f"{group_name_a} vs {group_name_b}",
            'Feature': col,
            'U_Statistic': u_stat,
            'p_value': p_val,
            'Rank_Biserial_r': r,
            'Mean_A': x.mean(),
            'Mean_B': y.mean()
        })
    return pd.DataFrame(results)

# Run comparisons required by the manuscript
res1 = run_mwu_tests(group_normal, df[df['IF_Label'] == -1], 'Normal', 'IF Anomalies (All)')
res2 = run_mwu_tests(group_normal, df[df['LOF_05_Label'] == -1], 'Normal', 'LOF Anomalies (All)')
res3 = run_mwu_tests(group_if_only, group_lof_only, 'IF-Only', 'LOF-Only')

df_table3 = pd.concat([res1, res2, res3], ignore_index=True)

# Export to CSV (Fixing BUG-03)
out_table3 = BASE_DIR / 'outputs' / 'tables' / 'table3_mannwhitney_effectsize.csv'
df_table3.to_csv(out_table3, index=False)

print("Table 3: Mann-Whitney U Results + Effect Sizes")
display(df_table3.head(10))\n

## 3. Chi-Square & Cramér's V (Categorical Features)\n

In [ ]:
# Cell 06: Run Chi-Square Tests
def cramers_v(chi2, n, shape):
    # Cramér's V = sqrt(chi2 / (n * min(r-1, c-1)))
    r, c = shape
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))

def run_chi2_tests(group_a, group_b, group_name_a, group_name_b):
    results = []
    # Temporarily combine to create a cross-tabulation
    temp_a = group_a.copy()
    temp_a['__Group'] = group_name_a
    temp_b = group_b.copy()
    temp_b['__Group'] = group_name_b
    df_combined = pd.concat([temp_a, temp_b])
    
    for col in cat_features:
        if col not in df_combined.columns:
            continue
        
        contingency_table = pd.crosstab(df_combined[col], df_combined['__Group'])
        
        # Chi-Square Test
        chi2, p_val, dof, _ = chi2_contingency(contingency_table)
        
        # Cramér's V
        n = contingency_table.sum().sum()
        v = cramers_v(chi2, n, contingency_table.shape)
        
        results.append({
            'Comparison': f"{group_name_a} vs {group_name_b}",
            'Feature': col,
            'Chi2_Statistic': chi2,
            'p_value': p_val,
            'Degrees_of_Freedom': dof,
            'Cramers_V': v
        })
    return pd.DataFrame(results)

res_c1 = run_chi2_tests(group_normal, df[df['IF_Label'] == -1], 'Normal', 'IF Anomalies (All)')
res_c2 = run_chi2_tests(group_normal, df[df['LOF_05_Label'] == -1], 'Normal', 'LOF Anomalies (All)')
res_c3 = run_chi2_tests(group_if_only, group_lof_only, 'IF-Only', 'LOF-Only')

df_table4 = pd.concat([res_c1, res_c2, res_c3], ignore_index=True)

# Export to CSV (Fixing BUG-03)
out_table4 = BASE_DIR / 'outputs' / 'tables' / 'table4_chisquare_effectsize.csv'
df_table4.to_csv(out_table4, index=False)

print("Table 4: Chi-Square Results + Effect Sizes")
display(df_table4.head())\n